## Threads in tkinter
Warum braucht man Threads in Tkinter?

Tkinter hat nur einen einzigen Haupt-Thread, der:

* la superficie recorre

* Buttons anklickbar macht

* Events verarbeitet

Beispiel ohne Thread:

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(label):
    """Diese Funktion läuft im Hintergrund-Thread"""
    for i in range(5):
        time.sleep(1)  # Lange Berechnung simulieren
        label.config(text=f"Fortschritt: {i+1}/5")


root = tk.Tk()
root.title("Tkinter + Threads Beispiel")

label = ttk.Label(root, text="Klicke auf Start...")
label.pack(pady=20)

button = ttk.Button(root, text="Start", command=lambda: lange_aufgabe(label))
button.pack(pady=10)

root.mainloop()


El resultado: toda la GUI se congela.

### Solución: Deslocalizar el trabajo en un hilo de fondo
Que contenga: P0-C-P1-P2-Thread(target=lange_tarea).start()-P3-P

se pueden externalizar tareas de tiempo en otro hilo:

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(label):
    """Diese Funktion läuft im Hintergrund-Thread"""
    for i in range(5):
        time.sleep(1)  # Lange Berechnung simulieren
        label.config(text=f"Fortschritt: {i+1}/5")

def start_thread(label):
    """Startet die lange Aufgabe in einem eigenen Thread"""
    thread = threading.Thread(target=lange_aufgabe, args=(label,))
    thread.start()

root = tk.Tk()
root.title("Tkinter + Threads Beispiel")

label = ttk.Label(root, text="Klicke auf Start...")
label.pack(pady=20)

button = ttk.Button(root, text="Start", command=lambda: start_thread(label))
button.pack(pady=10)

root.mainloop()


#### Breve y sencilla explicación

Tkinter arbeitet nur in einem Haupt-Thread.

Una larga tarea allí bloquearía el GUI → Ventana se congela.

Es por eso que empezamos un hilo que hace la larga tarea.

In [ ]:
threading.Thread(target=lange_aufgabe).start()


Pero el hilo no puede cambiar directamente el GUI.

Así que vamos a enviar la actualización de vuelta a Tkinter:

In [ ]:
label.after(0, ...)

De este modo, la etiqueta se actualiza de nuevo en el Threed GUI , de forma segura y correcta.

□ Importante: Tkinter no es una caja de seguridad de hilo

A Tkinter no le gustan los cambios directos de un hilo de trabajo. El ejemplo simple de arriba suele ser bueno, pero más limpio sería:

Enviar cambios desde el hilo per after() de nuevo a la GUI

La variante limpia:

In [ ]:
def lange_aufgabe(label):
    for i in range(5):
        time.sleep(1)
        label.after(0, lambda i=i: label.config(text=f"Fortschritt: {i+1}/5"))


→ after(0, ...) realiza el cambio GUI de forma segura en Tkinter-Thread.

### El ejemplo con un objeto amplio, la barra de progreso
□ GUI no congela una barra de progreso P0 se actualiza limpiamente •P1 • Thread trabaja en el fondo • P2 • Actualizaciones por after() de nuevo en la GUI P3 •

In [ ]:
import threading
import time
import tkinter as tk
from tkinter import ttk

def lange_aufgabe(progress):
    """Lange Aufgabe in einem Hintergrund-Thread"""
    for i in range(101):  # 0 bis 100 %
        time.sleep(0.05)  # Rechenzeit simulieren

        progress.after(0, lambda i=i: progress.configure(value=i))

def start_thread(progress):
    """Startet den Hintergrund-Thread"""
    threading.Thread(target=lange_aufgabe, args=(progress,), daemon=True).start()

root = tk.Tk()
root.title("Fortschrittsbalken + Thread")

# Fortschrittsbalken
progress = ttk.Progressbar(root, orient="horizontal", length=300, mode="determinate", maximum=100)
progress.pack(pady=20)

# Button zum Starten
button = ttk.Button(root, text="Start", command=lambda: start_thread(progress))
button.pack(pady=10)

root.mainloop()

### Queue


Ein Worker-Thread schreibt Nachrichten in eine Queue.

La GUI cancela la cola y escribe los mensajes en el campo de texto.

Así puedes ver directamente cómo separar Threads y GUI limpios.


In [ ]:
import threading
import time
import queue
import tkinter as tk
from tkinter import scrolledtext

def worker_task(q):
    """Worker-Thread schreibt Nachrichten in die Queue"""
    for i in range(1, 11):
        time.sleep(0.5)  # Arbeit simulieren
        q.put(f"Nachricht {i} vom Worker-Thread")
    q.put("Fertig!")  # Signal, dass alles erledigt ist

def start_thread(q):
    threading.Thread(target=worker_task, args=(q,), daemon=True).start()

def check_queue(q, text_widget):
    """Fragt die Queue ab und schreibt Nachrichten ins Textfeld"""
    try:
        while True:
            message = q.get_nowait()
            text_widget.insert(tk.END, message + "\n")
            text_widget.see(tk.END)  # Scrollt automatisch nach unten
    except queue.Empty:
        pass
    root.after(100, check_queue, q, text_widget)

root = tk.Tk()
root.title("Tkinter + Thread + Queue + Textfeld")

# Thread → GUI Kommunikation
q = queue.Queue()

# Textfeld für Ausgaben
text_area = scrolledtext.ScrolledText(root, width=40, height=10)
text_area.pack(padx=10, pady=10)

# Start-Button
button = tk.Button(root, text="Start Worker", command=lambda: start_thread(q))
button.pack(pady=5)

# Queue-Polling starten
root.after(100, check_queue, q, text_area)

root.mainloop()


Warum Queue? <br>

* Sie verhindert direkte GUI-Zugriffe aus Threads.

* Ella es 100% tread-asegurada.

* Sie funktioniert auch bei mehreren Events/Werten pro Tick.

* Ella es el patrón más limpio para el hilo de Tkinter.

<img style="float: center;" src="img/wbs-logo.jpg">


Autor: Dirk Maric
### Ilustraciones y fuentes
https://de.wikipedia.org/wiki/Python_(Idioma de programación) El logotipo de Pythan es una marca registrada de la Fundación de Software de Pyothon Todos los logotipos publicados en este sitio web, así como marcas comerciales, productos y marcas registradas, son propiedad de sus respectivas empresas © WBS TRAINING AG • Todos los derechos reservados

### Nutzungsrechte:
El uso de esta documentación está autorizado exclusivamente para fines de formación de WBS TRAINING AG. La transferencia a terceros, incluso en parte, así como la reproducción y difusión de todo tipo (procedimientos electrónicos y de otro tipo), incluidas las traducciones, sólo se permite con el consentimiento previo por escrito del titular del derecho.

### Herausgeber:

WBS TRAINING AG Lorenzweg 5 12099 Berlin Responsabilidad: Todos los contenidos están investigados y compilados con el máximo cuidado para la documentación de formación. Nos esforzamos por actualizar toda la información y los datos en curso. Sin embargo, pueden producirse errores (por ejemplo, desviaciones del hardware y del software descritos mediante actualizaciones a corto plazo), de modo que no podamos garantizar la total conformidad, exactitud y actualidad.